# LAX — One-Cell Colab Repro Setup

Purpose: rebuild the LAX Colab environment after a runtime reset using the GitHub repository, pinned Python packages, and the trained v2 adapter stored in Google Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_URL = 'https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git'
REPO_DIR = Path('/content/psychology-raft-rag-chatbot')
ADAPTER_PATH = Path('/content/drive/MyDrive/psychology-qwen-qlora-v2')

print('Adapter exists:', (ADAPTER_PATH / 'adapter_model.safetensors').exists())


In [ ]:
%cd /content
import os

if not (REPO_DIR / '.git').exists():
    !git clone $REPO_URL
else:
    %cd /content/psychology-raft-rag-chatbot
    !git pull origin main

%cd /content/psychology-raft-rag-chatbot
!git status --short


In [ ]:
%cd /content/psychology-raft-rag-chatbot
!python -m pip install -q --upgrade --upgrade-strategy eager -r requirements-colab.txt
!python -m pip check


In [ ]:
import torch, transformers, peft, bitsandbytes as bnb, streamlit

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('bitsandbytes:', bnb.__version__)
print('Streamlit:', streamlit.__version__)
print('v2 adapter:', (ADAPTER_PATH / 'adapter_model.safetensors').exists())


In [ ]:
from pathlib import Path

TOOLS_DIR = Path('/content/drive/MyDrive/LAX-tools')
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
DEB_PATH = TOOLS_DIR / 'cloudflared-linux-amd64.deb'

if not DEB_PATH.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O $DEB_PATH

!dpkg -i $DEB_PATH >/tmp/cloudflared_install.log 2>&1 || (cat /tmp/cloudflared_install.log; exit 1)
!cloudflared --version


## After setup
Run the final app with:

`!nohup streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.headless=true >/content/lax.log 2>&1 &`

Then start a Cloudflare Quick Tunnel to port 8501.
